## 8. Geração de Relatório de Comparação

### 8.1 Importação e Preparação dos Dados

In [ ]:
from src.report_generator import generate_report, format_for_display, save_report
from IPython.display import Markdown, display
import json

print("✅ Módulo report_generator importado com sucesso!")

### 8.2 Preparar Resultados da Comparação

In [ ]:
# Preparar lista de resultados de comparação para o relatório
comparison_results = []

# Processar seções alinhadas (modificadas)
if 'alignment_result' in globals() and alignment_result and 'alignments' in alignment_result:
    # CORREÇÃO: alignments é um DICIONÁRIO {section_a_id: {...}}, não lista
    for section_id_a, alignment_data in alignment_result['alignments'].items():
        section_id_b = alignment_data['section_b_id']
        confidence = alignment_data['confidence']
        
        # Simular resultado de comparação (em produção, viria do text_comparator)
        result = {
            "section_id": section_id_a,
            "section_title": alignment_data.get('title_a', f"Section {section_id_a}"),
            "type": "modified",
            "confidence": confidence,
            "severity": "MINOR" if confidence > 0.9 else "MEDIUM",
        }
        
        # Adicionar análise semântica se disponível
        if 'semantic_results' in globals() and semantic_results:
            # Procurar resultado semântico correspondente
            for sem_result in semantic_results:
                if sem_result.get('section_id') == section_id_a:
                    result['semantic_analysis'] = sem_result.get('analysis', {})
                    # Mapear classificação semântica para severidade
                    classification = sem_result.get('analysis', {}).get('classification')
                    if classification == 'SIGNIFICANT':
                        result['severity'] = 'CRITICAL'
                    elif classification == 'MINOR':
                        result['severity'] = 'MEDIUM'
                    break
        
        comparison_results.append(result)

# Adicionar seções não alinhadas (adicionadas/removidas)
if 'alignment_result' in globals() and alignment_result:
    # CORREÇÃO: 'added' e 'removed' são listas de dicionários
    for section_data in alignment_result.get('added', []):
        comparison_results.append({
            "section_id": section_data['id'],
            "section_title": section_data['title'],
            "type": "added",
            "severity": "SIGNIFICANT",
            "confidence": 1.0
        })
    
    for section_data in alignment_result.get('removed', []):
        comparison_results.append({
            "section_id": section_data['id'],
            "section_title": section_data['title'],
            "type": "removed",
            "severity": "SIGNIFICANT",
            "confidence": 1.0
        })

print(f"✅ Preparados {len(comparison_results)} resultados de comparação")
print(f"  - Modificadas: {sum(1 for r in comparison_results if r['type'] == 'modified')}")
print(f"  - Adicionadas: {sum(1 for r in comparison_results if r['type'] == 'added')}")
print(f"  - Removidas: {sum(1 for r in comparison_results if r['type'] == 'removed')}")

### 8.3 Gerar Relatório Completo

In [ ]:
# Gerar relatório de comparação
print("🔄 Gerando relatório de comparação...")

# Determinar nomes dos documentos
doc_a_name = pdf_paths[0] if 'pdf_paths' in globals() else "ASTM A29/A29M - 2015"
doc_b_name = pdf_paths[1] if 'pdf_paths' in globals() and len(pdf_paths) > 1 else "ASTM A29/A29M - 2016"

# Gerar relatório completo
report = generate_report(
    comparison_results=comparison_results,
    doc_a=doc_a_name,
    doc_b=doc_b_name,
    include_statistics=True,
    include_critical_analysis=True,
    max_critical_items=5
)

print(f"✅ Relatório gerado com sucesso!")
print(f"  - Tamanho: {len(report)} caracteres")
print(f"  - Linhas: {report.count(chr(10))}")

# Salvar relatório em arquivo
output_path = "data/outputs/comparison_report.md"
save_report(report, output_path)
print(f"  - Salvo em: {output_path}")

### 8.4 Visualizar Relatório Formatado

In [ ]:
# Exibir relatório formatado no notebook
formatted_report = format_for_display(report)
display(Markdown(formatted_report))

## 9. Pipeline Completa End-to-End

### 9.1 Executar Pipeline Completa de Comparação

In [ ]:
"""
PIPELINE COMPLETA DE COMPARAÇÃO DE DOCUMENTOS TÉCNICOS
======================================================
Este código executa todo o processo end-to-end de comparação entre dois PDFs.
"""

import time
from pathlib import Path

# Importar todos os módulos necessários
from src.config import setup_logging
from src.pdf_loader import load_pdf
from src.text_extractor import extract_text, parse_section_hierarchy
from src.table_extractor import extract_tables_from_pdf
from src.section_aligner import align_sections, get_section_by_id
from src.text_comparator import compare_text
from src.semantic_comparator import create_semantic_agent, classify_semantic_significance
from src.report_generator import generate_report, format_for_display, save_report
from IPython.display import Markdown, display

print("="*60)
print("FASTCHECKAI - PIPELINE COMPLETA DE COMPARAÇÃO")
print("="*60)

# Configurar logging
setup_logging(log_level="INFO")

# Medir tempo total
start_time = time.time()

# 1. CONFIGURAÇÃO
print("\n[1/8] 📁 Configuração dos Caminhos...")
pdf_dir = Path("data/inputs")
pdf_paths = [
    pdf_dir / "astm_2015.pdf",  
    pdf_dir / "astm_2016.pdf"
]

# Verificar existência dos arquivos
for path in pdf_paths:
    if path.exists():
        print(f"  ✅ {path.name} encontrado")
    else:
        print(f"  ❌ {path.name} NÃO encontrado")

print(f"\nTempo parcial: {time.time() - start_time:.2f}s")

In [ ]:
# 2. CARREGAR PDFs
print("\n[2/8] 📄 Carregando PDFs...")
pdf_docs = []
for path in pdf_paths:
    try:
        doc = load_pdf(str(path))
        pdf_docs.append(doc)
        print(f"  ✅ {path.name}: {len(doc)} páginas")
    except Exception as e:
        print(f"  ❌ Erro ao carregar {path.name}: {e}")
        
print(f"\nTempo parcial: {time.time() - start_time:.2f}s")

In [ ]:
# 3. EXTRAIR TEXTO E SEÇÕES
print("\n[3/8] 📝 Extraindo Texto e Seções...")

all_sections = []
for i, doc in enumerate(pdf_docs):
    print(f"  Processando {pdf_paths[i].name}...")
    
    # Extrair texto com OCR fallback se necessário
    text = extract_text(doc, enable_ocr=True)
    print(f"    - Texto extraído: {len(text)} caracteres")
    
    # Extrair seções hierárquicas
    sections = parse_section_hierarchy(text)
    all_sections.append(sections)
    
    # Contar seções
    total_sections = len(sections)
    total_subsections = sum(len(s.get('subsections', {})) for s in sections.values())
    print(f"    - Seções: {total_sections} principais, {total_subsections} subseções")

sections_a, sections_b = all_sections[0], all_sections[1]
print(f"\nTempo parcial: {time.time() - start_time:.2f}s")

In [ ]:
# 4. ALINHAR SEÇÕES
print("\n[4/8] 🔗 Alinhando Seções entre Documentos...")

alignment_result = align_sections(sections_a, sections_b, use_llm_fallback=False)

print(f"  ✅ Alinhamento concluído:")
print(f"    - Seções alinhadas: {len(alignment_result['alignments'])}")
print(f"    - Confiança média: {alignment_result['metadata']['avg_confidence']:.2%}")
print(f"    - Seções adicionadas: {len(alignment_result.get('added', []))}")
print(f"    - Seções removidas: {len(alignment_result.get('removed', []))}")

print(f"\nTempo parcial: {time.time() - start_time:.2f}s")

In [ ]:
# 5. COMPARAR TEXTO DAS SEÇÕES ALINHADAS
print("\n[5/8] 🔍 Comparando Conteúdo das Seções...")

text_diffs = []
critical_changes = 0

# CORREÇÃO: alignments é dicionário, iterar corretamente
sample_alignments = list(alignment_result['alignments'].items())[:5]  # Pegar primeiros 5
for section_id_a, alignment_data in sample_alignments:
    section_id_b = alignment_data['section_b_id']
    
    # Obter seções usando função auxiliar
    section_a = get_section_by_id(sections_a, section_id_a)
    section_b = get_section_by_id(sections_b, section_id_b)
    
    if section_a and section_b:
        # Comparar textos
        diff_result = compare_text(
            section_a.get('content', ''),
            section_b.get('content', '')
        )
        
        # Verificar termos críticos
        has_critical = any(term in str(diff_result.get('differences', [])) 
                          for term in ['shall', 'must', 'required', 'mandatory'])
        if has_critical:
            critical_changes += 1
            
        text_diffs.append({
            'section': section_id_a,
            'diff': diff_result,
            'critical': has_critical
        })

print(f"  ✅ Comparação concluída:")
print(f"    - Seções comparadas: {len(text_diffs)}")
print(f"    - Mudanças críticas detectadas: {critical_changes}")

print(f"\nTempo parcial: {time.time() - start_time:.2f}s")

In [ ]:
# 6. ANÁLISE SEMÂNTICA COM AGNO + GPT-4o
print("\n[6/8] 🧠 Análise Semântica com Agno Framework...")

# Criar agente semântico
try:
    agent = create_semantic_agent()
    print("  ✅ Agente Agno criado com sucesso")
except Exception as e:
    print(f"  ⚠️ Erro ao criar agente: {e}")
    agent = None

# Analisar mudanças significativas (limitar para demo)
semantic_results = []
if agent and text_diffs:
    for diff_info in text_diffs[:3]:  # Analisar apenas 3 para demo
        if diff_info['diff'].get('has_differences'):
            # Pegar primeira diferença como exemplo
            first_diff = diff_info['diff']['differences'][0] if diff_info['diff']['differences'] else None
            
            if first_diff:
                print(f"  Analisando seção {diff_info['section']}...")
                
                result = classify_semantic_significance(
                    {'original': first_diff.get('original', ''), 
                     'content': first_diff.get('modified', '')},
                    agent=agent
                )
                
                semantic_results.append({
                    'section_id': diff_info['section'],
                    'analysis': result
                })
                
                print(f"    - Classificação: {result.get('classification')}")
                print(f"    - Confiança: {result.get('confidence', 0):.1%}")

print(f"\n  ✅ Análise semântica concluída: {len(semantic_results)} seções analisadas")
print(f"\nTempo parcial: {time.time() - start_time:.2f}s")

In [ ]:
# 7. PREPARAR RESULTADOS PARA RELATÓRIO
print("\n[7/8] 📊 Preparando Resultados para Relatório...")

comparison_results = []

# CORREÇÃO: alignments é dicionário {section_a_id: {...}}, não lista
for section_id_a, alignment_data in alignment_result['alignments'].items():
    section_id_b = alignment_data['section_b_id']
    confidence = alignment_data['confidence']
    
    result = {
        "section_id": section_id_a,
        "section_title": alignment_data.get('title_a', f"Section {section_id_a}"),
        "type": "modified",
        "confidence": confidence,
        "severity": "MINOR"  # Default
    }
    
    # Adicionar análise semântica se disponível
    for sem_result in semantic_results:
        if sem_result['section_id'] == section_id_a:
            result['semantic_analysis'] = sem_result['analysis']
            # Mapear classificação para severidade
            classification = sem_result['analysis'].get('classification')
            if classification == 'SIGNIFICANT':
                result['severity'] = 'CRITICAL'
            elif classification == 'MINOR':
                result['severity'] = 'MEDIUM'
            break
    
    # Adicionar diff preview se disponível
    for diff_info in text_diffs:
        if diff_info['section'] == section_id_a:
            if diff_info['diff'].get('differences'):
                diff_preview = ""
                for d in diff_info['diff']['differences'][:2]:  # Primeiras 2 diferenças
                    diff_preview += f"- {d.get('original', '')[:50]}...\n"
                    diff_preview += f"+ {d.get('modified', '')[:50]}...\n"
                result['diff_preview'] = diff_preview
            
            # Adicionar termos críticos se detectados
            if diff_info.get('critical'):
                result['critical_terms'] = ['shall', 'must', 'required']
                result['severity'] = 'CRITICAL'
            break
    
    comparison_results.append(result)

# CORREÇÃO: 'added' e 'removed' são listas de dicionários
for section_data in alignment_result.get('added', []):
    comparison_results.append({
        "section_id": section_data['id'],
        "section_title": section_data['title'],
        "type": "added",
        "severity": "SIGNIFICANT",
        "confidence": 1.0
    })

for section_data in alignment_result.get('removed', []):
    comparison_results.append({
        "section_id": section_data['id'],
        "section_title": section_data['title'],
        "type": "removed",
        "severity": "SIGNIFICANT", 
        "confidence": 1.0
    })

print(f"  ✅ Resultados preparados:")
print(f"    - Total de mudanças: {len(comparison_results)}")
print(f"    - Críticas: {sum(1 for r in comparison_results if r.get('severity') == 'CRITICAL')}")
print(f"    - Significativas: {sum(1 for r in comparison_results if r.get('severity') == 'SIGNIFICANT')}")
print(f"    - Menores: {sum(1 for r in comparison_results if r.get('severity') == 'MINOR')}")

print(f"\nTempo parcial: {time.time() - start_time:.2f}s")

In [ ]:
# 8. GERAR RELATÓRIO FINAL
print("\n[8/8] 📄 Gerando Relatório Final...")

# Gerar relatório completo
report = generate_report(
    comparison_results=comparison_results,
    doc_a=str(pdf_paths[0]),
    doc_b=str(pdf_paths[1]),
    include_statistics=True,
    include_critical_analysis=True,
    max_critical_items=5
)

# Salvar relatório
output_path = "data/outputs/pipeline_complete_report.md"
save_report(report, output_path)

# Calcular tempo total
total_time = time.time() - start_time

print(f"  ✅ Relatório gerado com sucesso!")
print(f"    - Tamanho: {len(report)} caracteres")
print(f"    - Salvo em: {output_path}")

print("\n" + "="*60)
print("PIPELINE CONCLUÍDA COM SUCESSO!")
print("="*60)
print(f"⏱️  Tempo total de execução: {total_time:.2f} segundos")

# Verificar se atingimos o objetivo de < 3 minutos
if total_time < 180:
    print("✅ OBJETIVO ATINGIDO: Pipeline executada em menos de 3 minutos!")
else:
    print(f"⚠️  Tempo excedeu 3 minutos em {total_time - 180:.1f} segundos")

# Estatísticas finais
print("\n📊 Resumo da Execução:")
print(f"  - PDFs processados: 2")
print(f"  - Páginas totais: {sum(len(doc) for doc in pdf_docs)}")
print(f"  - Seções alinhadas: {len(alignment_result['alignments'])}")
print(f"  - Mudanças detectadas: {len(comparison_results)}")
print(f"  - Análises semânticas: {len(semantic_results)}")
print(f"  - Tempo médio por página: {total_time / sum(len(doc) for doc in pdf_docs):.2f}s")

# Validar viabilidade do Agno
if semantic_results:
    agno_sources = [r['analysis'].get('source', '') for r in semantic_results]
    fallback_rate = agno_sources.count('openai_fallback') / len(agno_sources) * 100 if agno_sources else 0
    
    print(f"\n🤖 Validação do Agno Framework:")
    print(f"  - Taxa de fallback: {fallback_rate:.1f}%")
    if fallback_rate < 30:
        print("  ✅ Agno Framework VIÁVEL para produção")
    else:
        print("  ⚠️ Taxa de fallback alta - considerar migração para OpenAI direto")

### 9.2 Exibir Relatório Final Formatado

In [ ]:
# Exibir relatório formatado
print("Exibindo relatório final formatado:\n")
print("="*60 + "\n")

formatted_report = format_for_display(report)
display(Markdown(formatted_report))

# ⚙️ Configuração Inicial

**IMPORTANTE**: Execute esta célula primeiro para configurar o ambiente do notebook.

In [ ]:
# Configuração do ambiente para importação de módulos
import sys
from pathlib import Path

# Adicionar diretório raiz do projeto ao Python path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"✅ Python path configurado: {project_root}")
print(f"✅ Diretório de trabalho: {Path().resolve()}")

# FastCheckAI - Pipeline de Comparação de PDFs

Este notebook implementa o pipeline completo de comparação de documentos técnicos em PDF usando processamento híbrido e análise semântica.

## Setup e Validação

Antes de começar, vamos validar que todas as dependências estão instaladas e o ambiente está configurado corretamente.

### 1.1 Validação de Imports

In [ ]:
# Validação de imports essenciais
import sys
import pymupdf
import pdfplumber
import pandas as pd
from agno.agent import Agent
from agno.models.openai.chat import OpenAIChat
from openai import OpenAI

print("✅ Todos os imports essenciais foram bem-sucedidos!")
print(f"Python version: {sys.version}")
print(f"PyMuPDF version: {pymupdf.__version__}")
print(f"pdfplumber version: {pdfplumber.__version__}")
print(f"pandas version: {pd.__version__}")

# Verificar versão do agno
import agno
print(f"agno version: {agno.__version__}")

### 1.2 Validação de Configuração (API Keys)

In [ ]:
# Validação de configuração e API keys
from src.config import (
    OPENAI_API_KEY,
    MODEL,
    TEMPERATURE,
    FUZZY_MATCH_THRESHOLD,
    SEMANTIC_CONFIDENCE_THRESHOLD,
    logger
)

print("✅ Configuração carregada com sucesso!")
print(f"Modelo: {MODEL}")
print(f"Temperature: {TEMPERATURE}")
print(f"Fuzzy Match Threshold: {FUZZY_MATCH_THRESHOLD}")
print(f"Semantic Confidence Threshold: {SEMANTIC_CONFIDENCE_THRESHOLD}")
print(f"API Key configurada: {'Sim' if OPENAI_API_KEY else 'Não'}")

### 1.3 Teste de Conexão OpenAI API

In [ ]:
# Teste de conexão com OpenAI API
from openai import OpenAI
import time

client = OpenAI(api_key=OPENAI_API_KEY)

start_time = time.time()
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Fale Olá em uma palavra"}],
    temperature=TEMPERATURE
)
elapsed_time = time.time() - start_time

print(f"✅ Conexão OpenAI API bem-sucedida!")
print(f"Resposta: {response.choices[0].message.content}")
print(f"Tempo de resposta: {elapsed_time:.2f}s")
print(f"Tokens usados: {response.usage.total_tokens}")

### 1.4 Validação Agno Framework

O Agno Framework será usado para orquestração de agentes LLM para análise semântica. Vamos validar que funciona corretamente.

**Notas importantes:**
- Importações corretas: 
  - `from agno.agent import Agent`
  - `from agno.models.openai.chat import OpenAIChat`
- O Agent requer um objeto Model, não uma string
- Criar modelo com: `OpenAIChat(id="gpt-4o")`

In [ ]:
# Teste Agno Framework - Hello World
from agno.agent import Agent
from agno.models.openai.chat import OpenAIChat
import time

# Criar modelo OpenAI
model = OpenAIChat(id=MODEL)

# Criar agente Agno com GPT-4o
agent = Agent(
    model=model,
    instructions="You are a helpful assistant specialized in technical documentation.",
)

# Teste simples
start_time = time.time()
response = agent.run("Explain what a technical standard is in one sentence")
elapsed_time = time.time() - start_time

print("✅ Agno Framework validado com sucesso!")
print(f"Resposta do agente: {response.content}")
print(f"Tempo de resposta: {elapsed_time:.2f}s")

---

## ✅ Setup Completo!

Se todas as células acima foram executadas sem erros, seu ambiente está pronto para desenvolvimento.

**Próximos passos:**
1. Implementar pipeline de extração de PDF (Feature 2)
2. Desenvolver sistema de alinhamento heurístico (Feature 3)
3. Integrar análise semântica com Agno (Feature 4)

---

## 2. Carregamento e Validação de PDFs

Esta seção implementa o carregamento robusto dos dois PDFs a serem comparados com:
- Validação de tamanho (≤25MB)
- Detecção automática de tipo (nativo vs escaneado)
- Tratamento de erros claros

### 2.1 Carregamento dos PDFs

In [ ]:
PDF_PATH_01 = '../data/ASTM_A29_A29M_Rev.00\'2015.pdf'
PDF_PATH_02 = '../data/ASTM_A29_A29M_Rev.00\'2016.pdf'

In [ ]:
# Importar módulo de carregamento de PDFs
from src.pdf_loader import load_pdf, detect_pdf_type

# Carregar os dois PDFs para comparação
print("📄 Carregando PDFs...")
pdf_a = load_pdf(PDF_PATH_01)
pdf_b = load_pdf(PDF_PATH_02)

print(f"\n✅ PDF A carregado: {pdf_a.page_count} páginas")
print(f"✅ PDF B carregado: {pdf_b.page_count} páginas")

### 2.2 Detecção de Tipo de PDF

In [ ]:
# Detectar tipo dos PDFs (nativo vs escaneado)
print("🔍 Detectando tipo dos PDFs...\n")

type_a = detect_pdf_type(pdf_a)
type_b = detect_pdf_type(pdf_b)

print(f"PDF A: {'✅ Nativo (texto extraível)' if type_a['is_native'] else '⚠️  Escaneado (requer OCR)'}")
print(f"  └─ {type_a['char_count']} caracteres na página 1")

print(f"\nPDF B: {'✅ Nativo (texto extraível)' if type_b['is_native'] else '⚠️  Escaneado (requer OCR)'}")
print(f"  └─ {type_b['char_count']} caracteres na página 1")

# Avisar se algum PDF requer OCR
if type_a['requires_ocr'] or type_b['requires_ocr']:
    print("\n⚠️  ATENÇÃO: Um ou mais PDFs parecem ser escaneados e podem requerer OCR para extração completa.")

---

## 3. Extração de Texto e Parsing de Estrutura

Esta seção implementa a extração completa de texto preservando hierarquia de seções e metadados.

### 3.1 Extração de Metadados dos PDFs

In [ ]:
# Extrair metadados dos PDFs
from src.text_extractor import extract_metadata

print("📊 Extraindo metadados dos PDFs...\n")

metadata_a = extract_metadata(pdf_a)
metadata_b = extract_metadata(pdf_b)

print("PDF A:")
for key, value in metadata_a.items():
    print(f"  {key}: {value}")

print("\nPDF B:")
for key, value in metadata_b.items():
    print(f"  {key}: {value}")

### 3.2 Extração do texto completo

In [ ]:
# Extrair texto completo dos PDFs com marcadores de página
from src.text_extractor import extract_text
import time

print("📄 Extraindo texto dos PDFs...\n")

# PDF A
start_time = time.time()
text_a = extract_text(pdf_a, include_page_markers=True)
time_a = time.time() - start_time

print(f"✅ PDF A: {len(text_a)} caracteres extraídos em {time_a:.2f}s")
print(f"   Taxa: {len(text_a)/time_a:.0f} chars/s")
print(f"   Primeiros 200 caracteres: {text_a[:200]}...")

# PDF B
print()
start_time = time.time()
text_b = extract_text(pdf_b, include_page_markers=True)
time_b = time.time() - start_time

print(f"✅ PDF B: {len(text_b)} caracteres extraídos em {time_b:.2f}s")
print(f"   Taxa: {len(text_b)/time_b:.0f} chars/s")
print(f"   Primeiros 200 caracteres: {text_b[:200]}...")

### 3.3 Parsing de Hierarquia de Seções

In [ ]:
# Fazer parsing da hierarquia de seções
from src.text_extractor import parse_section_hierarchy

print("🔍 Fazendo parsing da hierarquia de seções...\n")

# PDF A
sections_a = parse_section_hierarchy(text_a)
print(f"PDF A: {len(sections_a)} seções de nível 1")
print(f"Seções encontradas: {list(sections_a.keys())[:10]}")  # Primeiras 10

# Mostrar exemplo de estrutura hierárquica
if sections_a:
    first_section_id = list(sections_a.keys())[0]
    first_section = sections_a[first_section_id]
    print(f"\nExemplo de seção (ID: {first_section_id}):")
    print(f"  Título: {first_section['title']}")
    print(f"  Nível: {first_section['level']}")
    print(f"  Conteúdo (primeiros 150 chars): {first_section['content'][:150]}...")
    print(f"  Subseções: {list(first_section['subsections'].keys())}")

print("\n" + "="*60 + "\n")

# PDF B
sections_b = parse_section_hierarchy(text_b)
print(f"PDF B: {len(sections_b)} seções de nível 1")
print(f"Seções encontradas: {list(sections_b.keys())[:10]}")  # Primeiras 10

# Mostrar exemplo de estrutura hierárquica
if sections_b:
    first_section_id = list(sections_b.keys())[0]
    first_section = sections_b[first_section_id]
    print(f"\nExemplo de seção (ID: {first_section_id}):")
    print(f"  Título: {first_section['title']}")
    print(f"  Nível: {first_section['level']}")
    print(f"  Conteúdo (primeiros 150 chars): {first_section['content'][:150]}...")
    print(f"  Subseções: {list(first_section['subsections'].keys())}")

### 3.4 Estatísticas de Extração

In [ ]:
# Calcular estatísticas de extração
def count_sections_recursive(sections_dict):
    """Conta total de seções recursivamente"""
    count = len(sections_dict)
    for section in sections_dict.values():
        count += count_sections_recursive(section.get('subsections', {}))
    return count

print("📊 Estatísticas de Extração\n")
print("="*60)

print(f"\nPDF A:")
print(f"  Páginas: {metadata_a['page_count']}")
print(f"  Caracteres extraídos: {len(text_a):,}")
print(f"  Seções nível 1: {len(sections_a)}")
print(f"  Total de seções (todos níveis): {count_sections_recursive(sections_a)}")
print(f"  Tempo de extração: {time_a:.2f}s")

print(f"\nPDF B:")
print(f"  Páginas: {metadata_b['page_count']}")
print(f"  Caracteres extraídos: {len(text_b):,}")
print(f"  Seções nível 1: {len(sections_b)}")
print(f"  Total de seções (todos níveis): {count_sections_recursive(sections_b)}")
print(f"  Tempo de extração: {time_b:.2f}s")

print("\n" + "="*60)
print("✅ Extração de texto completa! Dados prontos para alinhamento e comparação.")

### 3.5 Testes de Validação

Vamos testar os casos de erro para garantir que as validações estão funcionando corretamente.

In [ ]:
# Teste 1: Validação de tamanho (PDF muito grande)
print("🧪 Teste 1: Validação de tamanho máximo\n")
try:
    large_pdf = load_pdf("data/inputs/large_file.pdf")
    print("❌ FALHA: PDF grande deveria ter sido rejeitado")
except Exception as e:
    print(f"✅ SUCESSO: {type(e).__name__}: {e}\n")

# Teste 2: PDF corrompido
print("🧪 Teste 2: Tratamento de PDF corrompido\n")
try:
    corrupt_pdf = load_pdf("data/inputs/corrupted.pdf")
    print("❌ FALHA: PDF corrompido deveria ter gerado erro")
except Exception as e:
    print(f"✅ SUCESSO: {type(e).__name__}: {e}\n")

# Teste 3: Arquivo inexistente
print("🧪 Teste 3: Tratamento de arquivo inexistente\n")
try:
    missing_pdf = load_pdf("data/inputs/nao_existe.pdf")
    print("❌ FALHA: Arquivo inexistente deveria gerar erro")
except FileNotFoundError as e:
    print(f"✅ SUCESSO: FileNotFoundError: {e}\n")

# Teste 4: PDF escaneado (sem texto)
print("🧪 Teste 4: Detecção de PDF escaneado\n")
try:
    scanned_pdf = load_pdf("data/inputs/scanned_sample.pdf")
    scanned_type = detect_pdf_type(scanned_pdf)
    if scanned_type['requires_ocr']:
        print(f"✅ SUCESSO: PDF detectado como escaneado ({scanned_type['char_count']} chars)")
    else:
        print(f"❌ FALHA: PDF deveria ser detectado como escaneado")
    scanned_pdf.close()
except Exception as e:
    print(f"❌ ERRO: {e}")

print("\n✅ Todos os testes de validação concluídos!")

---

## 4. Detecção e Extração de Tabelas

Esta seção implementa a extração de tabelas dos PDFs usando:
- PyMuPDF para detecção rápida de páginas com tabelas
- pdfplumber para extração precisa de dados tabulares
- Metadados ricos para posterior alinhamento entre documentos

### 4.1 Detecção de Páginas com Tabelas

In [ ]:
# Detectar páginas que contêm tabelas
from src.table_extractor import detect_table_pages
import time

print("🔍 Detectando páginas com tabelas...\n")

# PDF A
start_time = time.time()
table_pages_a = detect_table_pages(pdf_a)
time_a = time.time() - start_time

print(f"✅ PDF A: {len(table_pages_a)} páginas com tabelas detectadas em {time_a:.2f}s")
print(f"   Páginas: {table_pages_a}")

# PDF B
print()
start_time = time.time()
table_pages_b = detect_table_pages(pdf_b)
time_b = time.time() - start_time

print(f"✅ PDF B: {len(table_pages_b)} páginas com tabelas detectadas em {time_b:.2f}s")
print(f"   Páginas: {table_pages_b}")

### 4.2 Extração de Tabelas

In [ ]:
# Extrair tabelas das páginas detectadas
from src.table_extractor import extract_tables

print("📊 Extraindo tabelas dos PDFs...\n")

# PDF A
if table_pages_a:
    result_a = extract_tables(PDF_PATH_01, table_pages_a)
    print(result_a.summary())
    print(f"\n  Tabelas extraídas: {result_a.total_tables_extracted}")
    print(f"  Páginas com falha: {len(result_a.failed_pages)}")
else:
    print("PDF A: Nenhuma página com tabelas detectada")
    result_a = None

print("\n" + "="*60 + "\n")

# PDF B
if table_pages_b:
    result_b = extract_tables(PDF_PATH_02, table_pages_b)
    print(result_b.summary())
    print(f"\n  Tabelas extraídas: {result_b.total_tables_extracted}")
    print(f"  Páginas com falha: {len(result_b.failed_pages)}")
else:
    print("PDF B: Nenhuma página com tabelas detectada")
    result_b = None

### 4.3 Visualização das Tabelas Extraídas

In [ ]:
# Mostrar as primeiras tabelas extraídas
print("📋 Visualizando tabelas extraídas...\n")

# PDF A - Mostrar primeiras 3 tabelas
if result_a and result_a.tables:
    print(f"PDF A - Primeiras {min(3, len(result_a.tables))} tabelas:\n")
    
    for i, table in enumerate(result_a.tables[:3]):
        print(f"Tabela {i+1} (Página {table.metadata.page_num + 1}):")
        print(f"  Dimensão: {table.metadata.row_count}x{table.metadata.column_count}")
        print(f"  Header: {'Sim' if table.metadata.has_header else 'Não'}")
        print(f"  Confiança: {table.metadata.confidence:.2f}")
        print(f"  Seção: {table.metadata.section_context or 'N/A'}")
        print("\nPrimeiras linhas:")
        display(table.data.head(5))
        print("\n" + "-"*60 + "\n")
else:
    print("PDF A: Nenhuma tabela extraída\n")

print("="*60 + "\n")

# PDF B - Mostrar primeiras 3 tabelas
if result_b and result_b.tables:
    print(f"PDF B - Primeiras {min(3, len(result_b.tables))} tabelas:\n")
    
    for i, table in enumerate(result_b.tables[:3]):
        print(f"Tabela {i+1} (Página {table.metadata.page_num + 1}):")
        print(f"  Dimensão: {table.metadata.row_count}x{table.metadata.column_count}")
        print(f"  Header: {'Sim' if table.metadata.has_header else 'Não'}")
        print(f"  Confiança: {table.metadata.confidence:.2f}")
        print(f"  Seção: {table.metadata.section_context or 'N/A'}")
        print("\nPrimeiras linhas:")
        display(table.data.head(5))
        print("\n" + "-"*60 + "\n")
else:
    print("PDF B: Nenhuma tabela extraída")

### 4.4 Estatísticas de Extração de Tabelas

In [ ]:
# Calcular estatísticas de extração de tabelas
print("📊 Estatísticas de Extração de Tabelas\n")
print("="*60)

if result_a:
    print(f"\nPDF A:")
    print(f"  Páginas com tabelas: {result_a.total_pages_detected}")
    print(f"  Tabelas extraídas: {result_a.total_tables_extracted}")
    print(f"  Páginas com falha: {len(result_a.failed_pages)}")
    print(f"  Tempo de detecção: {result_a.detection_time_seconds:.2f}s")
    print(f"  Tempo de extração: {result_a.extraction_time_seconds:.2f}s")
    print(f"  Tempo total: {result_a.detection_time_seconds + result_a.extraction_time_seconds:.2f}s")
    
    if result_a.tables:
        avg_confidence = sum(t.metadata.confidence for t in result_a.tables) / len(result_a.tables)
        print(f"  Confiança média: {avg_confidence:.2f}")
else:
    print(f"\nPDF A: Nenhuma tabela detectada")

if result_b:
    print(f"\nPDF B:")
    print(f"  Páginas com tabelas: {result_b.total_pages_detected}")
    print(f"  Tabelas extraídas: {result_b.total_tables_extracted}")
    print(f"  Páginas com falha: {len(result_b.failed_pages)}")
    print(f"  Tempo de detecção: {result_b.detection_time_seconds:.2f}s")
    print(f"  Tempo de extração: {result_b.extraction_time_seconds:.2f}s")
    print(f"  Tempo total: {result_b.detection_time_seconds + result_b.extraction_time_seconds:.2f}s")
    
    if result_b.tables:
        avg_confidence = sum(t.metadata.confidence for t in result_b.tables) / len(result_b.tables)
        print(f"  Confiança média: {avg_confidence:.2f}")
else:
    print(f"\nPDF B: Nenhuma tabela detectada")

print("\n" + "="*60)
print("✅ Extração de tabelas completa! Dados prontos para comparação.")

---

## 5. Alinhamento de Seções

Esta seção alinha seções correspondentes entre os dois PDFs usando estratégia hierárquica:
1. **Exact match**: Alinhamento por ID de seção idêntico
2. **Fuzzy match**: Alinhamento por similaridade de título (threshold ≥ 0.8)
3. **LLM suggestion**: Agno sugere alinhamento para casos ambíguos

### 5.1 Alinhamento Heurístico de Seções

In [ ]:
# Importar função COMPLETA de alinhamento
from src.section_aligner import align_sections
import time

print("🔗 Alinhando seções entre PDFs...")
start_time = time.time()

# Executar alinhamento COMPLETO (heuristic + detecção de unmatched)
# use_llm_fallback=False para execução rápida (sem chamadas LLM)
alignment_result = align_sections(sections_a, sections_b, use_llm_fallback=False)
elapsed = time.time() - start_time

print(f"✅ Alinhamento concluído em {elapsed:.2f}s\n")
print(f"Estatísticas:")
print(f"  Seções alinhadas: {alignment_result['metadata']['aligned_count']}")
print(f"  Seções adicionadas (B): {alignment_result['metadata']['added_count']}")
print(f"  Seções removidas (A): {alignment_result['metadata']['removed_count']}")
print(f"  Confiança média: {alignment_result['metadata']['avg_confidence']:.2f}")

# Contar tipos de alinhamento
exact_matches = sum(1 for a in alignment_result['alignments'].values() if a['method'] == 'exact')
fuzzy_matches = sum(1 for a in alignment_result['alignments'].values() if a['method'] == 'fuzzy')

print(f"\nTipos de alinhamento:")
print(f"  Exact match: {exact_matches}")
print(f"  Fuzzy match: {fuzzy_matches}")

### 5.2 Visualização de Alinhamentos

In [ ]:
# Mostrar primeiros 10 pares alinhados
print("📋 Primeiros 10 pares de seções alinhadas:\n")
print("="*80)

# Converter alignments dict para lista para facilitar iteração
alignments_list = list(alignment_result['alignments'].items())

for i, (section_id_a, alignment_data) in enumerate(alignments_list[:10], 1):
    section_id_b = alignment_data['section_b_id']
    print(f"\n{i}. {section_id_a} ↔ {section_id_b}")
    print(f"   Tipo: {alignment_data['method']} | Confiança: {alignment_data['confidence']:.2f}")
    print(f"   Título A: {alignment_data['title_a'][:60]}...")
    print(f"   Título B: {alignment_data['title_b'][:60]}...")

# Mostrar seções não correspondidas
print(f"\n{'='*80}")
print(f"\n⚠️  Seções removidas do PDF A ({len(alignment_result['removed'])}):")
for section_info in alignment_result['removed'][:5]:
    print(f"  - {section_info['id']}: {section_info['title'][:50]}...")

print(f"\n⚠️  Seções adicionadas no PDF B ({len(alignment_result['added'])}):")
for section_info in alignment_result['added'][:5]:
    print(f"  - {section_info['id']}: {section_info['title'][:50]}...")

print(f"\n{'='*80}")
print("✅ Alinhamento de seções completo! Dados prontos para comparação textual.")

---

## 6. Comparação Textual

Esta seção compara o texto de seções alinhadas usando difflib:
- Detecta adições, remoções e modificações
- Filtra diferenças triviais (whitespace-only)
- Destaca mudanças numéricas (valores com unidades)
- Prioriza termos críticos (mandatory, shall, etc.)

### 6.1 Comparação de Seções Alinhadas

In [ ]:
# Importar módulo de comparação textual e função auxiliar
from src.text_comparator import compare_text
from src.section_aligner import get_section_by_id  # Função para buscar seções na hierarquia
import time

print("📝 Comparando texto de seções alinhadas...")
print("="*80)

# Comparar primeiras 5 seções alinhadas
comparison_results = []

# Converter alignments para lista
alignments_list = list(alignment_result['alignments'].items())

for i, (section_id_a, alignment_data) in enumerate(alignments_list[:5], 1):
    section_id_b = alignment_data['section_b_id']
    
    # CORREÇÃO: Usar função helper para buscar seções em qualquer nível da hierarquia
    section_a = get_section_by_id(sections_a, section_id_a)
    section_b = get_section_by_id(sections_b, section_id_b)
    
    # Verificação defensiva: pular se seção não for encontrada
    if section_a is None:
        print(f"\n{i}. ⚠️ AVISO: Seção {section_id_a} não encontrada no PDF A - pulando...")
        continue
    if section_b is None:
        print(f"\n{i}. ⚠️ AVISO: Seção {section_id_b} não encontrada no PDF B - pulando...")
        continue
    
    print(f"\n{i}. Comparando {section_id_a} ↔ {section_id_b}")
    
    start_time = time.time()
    diff_result = compare_text(section_a['content'], section_b['content'])
    elapsed = time.time() - start_time
    
    total_diffs = sum(len(v) for v in diff_result.values())
    
    print(f"   Tempo: {elapsed:.3f}s")
    print(f"   Diferenças: {total_diffs} total")
    print(f"   - Adições: {len(diff_result['additions'])}")
    print(f"   - Remoções: {len(diff_result['removals'])}")
    print(f"   - Modificações: {len(diff_result['modifications'])}")
    
    # Contar diffs flagados
    numeric_count = sum(1 for d in diff_result['modifications'] if d.get('contains_numeric_change'))
    critical_count = sum(1 for d in diff_result['modifications'] if d.get('contains_critical_term'))
    
    if numeric_count > 0 or critical_count > 0:
        print(f"   🚨 Flags: {numeric_count} numéricos, {critical_count} críticos")
    
    comparison_results.append({
        'section_id_a': section_id_a,
        'section_id_b': section_id_b,
        'diffs': diff_result,
        'total_diffs': total_diffs,
        'time': elapsed
    })

print(f"\n{'='*80}")
print(f"✅ Comparação textual de {len(comparison_results)} seções completa!")

### 6.2 Visualização de Diferenças Detectadas

In [ ]:
# Visualizar diferenças da primeira seção comparada
if comparison_results:
    print("📋 Exemplo de diferenças detectadas (primeira seção):\n")
    print("="*80)
    
    first_result = comparison_results[0]
    diffs = first_result['diffs']
    section_id_a = first_result['section_id_a']
    section_id_b = first_result['section_id_b']
    
    # Buscar títulos das seções diretamente do alignment_result
    title_a = alignment_result['alignments'][section_id_a]['title_a']
    
    print(f"Seção: {section_id_a} ↔ {section_id_b}")
    print(f"Título: {title_a[:60]}...\n")
    
    # Mostrar modificações (primeiras 3)
    if diffs['modifications']:
        print(f"Modificações ({len(diffs['modifications'])} total):\n")
        for i, mod in enumerate(diffs['modifications'][:3], 1):
            flags = []
            if mod.get('contains_numeric_change'):
                flags.append('🔢 NUMÉRICO')
            if mod.get('contains_critical_term'):
                flags.append('⚠️ CRÍTICO')
            
            flag_str = ' '.join(flags) if flags else ''
            print(f"{i}. {flag_str}")
            print(f"   Original: '{mod['original'][:80]}...'")
            print(f"   Novo:     '{mod['content'][:80]}...'")
            print()
    
    # Mostrar adições (primeiras 2)
    if diffs['additions']:
        print(f"\nAdições ({len(diffs['additions'])} total):\n")
        for i, add in enumerate(diffs['additions'][:2], 1):
            print(f"{i}. Adicionado: '{add['content'][:100]}...'")
    
    # Mostrar remoções (primeiras 2)
    if diffs['removals']:
        print(f"\nRemoções ({len(diffs['removals'])} total):\n")
        for i, rem in enumerate(diffs['removals'][:2], 1):
            print(f"{i}. Removido: '{rem['original'][:100]}...'")
    
    print(f"\n{'='*80}")

# Estatísticas gerais
print(f"\n📊 Estatísticas de Comparação Textual:\n")
print(f"  Seções comparadas: {len(comparison_results)}")
print(f"  Tempo total: {sum(r['time'] for r in comparison_results):.2f}s")
print(f"  Tempo médio por seção: {sum(r['time'] for r in comparison_results) / len(comparison_results):.3f}s")
print(f"  Total de diferenças: {sum(r['total_diffs'] for r in comparison_results)}")

print(f"\n{'='*80}")
print("✅ Diferenças textuais detectadas! Prontas para análise semântica.")

---

## 7. Análise Semântica com Agno Framework

Esta seção **valida o objetivo primário do PoC**: integração do Agno Framework para classificação semântica de diferenças detectadas.

**Critério de sucesso**: Taxa de fallback <30% (Agno viável se ≥70% das chamadas funcionarem)

### 7.1 Criação do Agente Semântico

In [ ]:
# Importar módulo de análise semântica
from src.semantic_comparator import create_semantic_agent, classify_semantic_significance, get_semantic_stats

import time

# Criar agente Agno para análise semântica
print("🤖 Criando agente semântico com Agno Framework...")
start_time = time.time()

agent = create_semantic_agent()
elapsed = time.time() - start_time

print(f"✅ Agente criado com sucesso em {elapsed:.2f}s")
print(f"\nConfiguração:")
print(f"  Modelo: {MODEL}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Instruções: Especialista em análise de normas técnicas ASTM")

### 7.2 Teste Básico do Agente (Hello World)

In [ ]:
# Teste simples para validar que o agente está funcionando
print("🧪 Teste 1: Hello World com Agno")
print("-" * 60)

start_time = time.time()
response = agent.run("What is semantic equivalence in one sentence?")
elapsed = time.time() - start_time

print(f"✅ Resposta recebida em {elapsed:.2f}s:")
print(f"\n{response.content}\n")

### 7.3 Testes de Classificação Semântica

Vamos testar os 3 níveis de classificação: EQUIVALENT, MINOR, SIGNIFICANT

In [ ]:
# Definir casos de teste para cada nível de significância
test_cases = [
    {
        "name": "EQUIVALENT - Reformulação sem mudança de sentido",
        "diff": {"original": "automobile", "content": "vehicle"},
        "expected": "EQUIVALENT"
    },
    {
        "name": "MINOR - Clarificação editorial",
        "diff": {"original": "shall be tested", "content": "shall be tested for compliance"},
        "expected": "MINOR"
    },
    {
        "name": "SIGNIFICANT - Mudança de requisito crítico",
        "diff": {"original": "mandatory testing", "content": "optional testing"},
        "expected": "SIGNIFICANT"
    },
    {
        "name": "SIGNIFICANT - Mudança numérica crítica",
        "diff": {"original": "tensile strength ≥ 500 MPa", "content": "tensile strength ≥ 550 MPa"},
        "expected": "SIGNIFICANT"
    }
]

print("🧪 Testes de Classificação Semântica")
print("=" * 80)

results = []

for i, test in enumerate(test_cases, 1):
    print(f"\nTeste {i}: {test['name']}")
    print("-" * 80)
    print(f"Original: '{test['diff']['original']}'")
    print(f"Modificado: '{test['diff']['content']}'")
    print(f"Esperado: {test['expected']}")
    
    start_time = time.time()
    result = classify_semantic_significance(test['diff'], agent)
    elapsed = time.time() - start_time
    
    status = "✅" if result['classification'] == test['expected'] else "⚠️ "
    print(f"\n{status} Resultado: {result['classification']} (confiança: {result['confidence']:.2f})")
    print(f"Fonte: {result['source']}")
    print(f"Reasoning: {result['reasoning']}")
    print(f"Tempo: {elapsed:.2f}s")
    
    results.append({
        "test": test['name'],
        "expected": test['expected'],
        "got": result['classification'],
        "match": result['classification'] == test['expected'],
        "confidence": result['confidence'],
        "time": elapsed,
        "source": result['source']
    })

# Resumo
print("\n" + "=" * 80)
print("RESUMO DOS TESTES")
print("=" * 80)

correct = sum(1 for r in results if r['match'])
total = len(results)
accuracy = correct / total * 100

print(f"\nAcurácia: {correct}/{total} ({accuracy:.1f}%)")
print(f"Tempo médio: {sum(r['time'] for r in results) / total:.2f}s")

agno_calls = sum(1 for r in results if r['source'] == 'agno')
fallback_calls = sum(1 for r in results if r['source'] == 'openai_fallback')
print(f"\nChamadas Agno: {agno_calls}")
print(f"Fallbacks OpenAI: {fallback_calls}")

if fallback_calls / total > 0.3:
    print(f"\n⚠️  ATENÇÃO: Taxa de fallback ({fallback_calls/total*100:.1f}%) excede 30%!")
else:
    print(f"\n✅ Agno viável: {fallback_calls/total*100:.1f}% fallback (target: <30%)")

### 7.4 Estatísticas e Custo da Análise Semântica

In [ ]:
# Obter estatísticas da sessão de análise semântica
from src.semantic_comparator import log_semantic_summary

print("📊 Estatísticas da Análise Semântica")
print("=" * 80)

stats = get_semantic_stats()

print(f"\nChamadas LLM:")
print(f"  Total: {stats['total_llm_calls']}")
print(f"  Agno: {stats['agno_calls']} ({100 - stats['fallback_rate']:.1f}%)")
print(f"  OpenAI fallback: {stats['fallback_calls']} ({stats['fallback_rate']:.1f}%)")
print(f"  Cache hits: {stats['cache_hits']}")

print(f"\nCusto estimado: ${stats['total_cost']:.4f}")

print(f"\n{'='*80}")
print("VALIDAÇÃO DO PoC - VIABILIDADE DO AGNO FRAMEWORK")
print("="*80)

if stats['fallback_rate'] <= 30.0:
    print(f"\n✅ SUCESSO: Agno é VIÁVEL para produção")
    print(f"   Taxa de fallback: {stats['fallback_rate']:.1f}% (target: <30%)")
    print(f"   {100 - stats['fallback_rate']:.1f}% das chamadas funcionaram com Agno")
else:
    print(f"\n⚠️  ATENÇÃO: Taxa de fallback elevada")
    print(f"   Taxa de fallback: {stats['fallback_rate']:.1f}% (target: <30%)")
    print(f"   Considerar migração para OpenAI SDK direto")

# Log completo
print(f"\n{'='*80}")
log_semantic_summary()